In [24]:
import sys
sys.path.append("../preprocess")
from qbiolip import load_db
import glob
import torch

df = load_db("../../databases/QBioLiP/qbiolip.csv", "../../databases/ec_uniprot_ids.txt")
df = df.loc[df.target]

output_path = "../../databases/Vina/*"
outputs = glob.glob(output_path)

names = []
targets = []
preds = []
scores = []

for output in outputs:
    name = output.split("/")[-1].split(".")[0]
    files = glob.glob(f"{output}/*.pdb")
    if len(files) == 0:
        continue
    output_preds = []
    output_scores = []
    target_file = f"../../databases/QBioLiP/nr_lig/{name.replace('_trimed', '')}.pdb"
    with open(target_file) as target_f:
        target_lines = [l for l in target_f]
        target = torch.tensor([
            [
                float(r[30:38].strip()), #x
                float(r[38:46].strip()), #y
                float(r[46:54].strip())  #z
            ] for r in target_lines if r[:6].strip() == "HETATM"
        ])
    for file in files:
        with open(file) as pdb_f:
            pred_lines = [l for l in pdb_f]

        score = pred_lines[1].split()[3:]
        score = torch.tensor([float(s) for s in score])
        pred = torch.tensor([
            [
                float(r[30:38].strip()), #x
                float(r[38:46].strip()), #y
                float(r[46:54].strip())  #z
            ] for r in pred_lines if r[:6].strip() == "HETATM"
        ])
        output_preds.append(pred)
        output_scores.append(score)
    output_preds = torch.stack(output_preds)
    output_scores = torch.stack(output_scores)
    names.append(name)
    preds.append(output_preds)
    targets.append(target)
    scores.append(output_scores)

Len of the database: 25415
Columns: id, pdb_id, assembly_id, stoichiometry, resolution, uniprot_id, assembly_detail, ligand_id, site, ligand_detail, pubmed_id, interaction, relevant, binding_site, binding_site_pdb, bindingMOAD, pdbbind_cn, binding_db, group, lig_canon_smiles, lig_atoms, rec_len_resi, centroid_x, centroid_y, centroid_z, x_size, y_size, z_size, target, exp_type, size_x, size_y, size_z
385
385
385
385
